# STEP Files

Open every tracked STEP artifact in VS Code with the OCP CAD Viewer extension.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import subprocess

import build123d as bd
from ocp_vscode import show


@dataclass(frozen=True)
class StepArtifact:
    label: str
    step_path: Path
    generator_source: Path


def require_repo_root() -> Path:
    result = subprocess.run(
        ["git", "rev-parse", "--show-toplevel"],
        check=True,
        capture_output=True,
        text=True,
    )
    root_text = result.stdout.strip()
    if not root_text:
        raise RuntimeError("git rev-parse --show-toplevel returned empty stdout")
    repo_root = Path(root_text).resolve()
    pyproject_path = repo_root / "pyproject.toml"
    if not pyproject_path.is_file():
        raise FileNotFoundError(f"repo root is missing pyproject.toml: {pyproject_path}")
    return repo_root


def require_file(repo_root: Path, repo_relative_path: Path) -> Path:
    absolute_path = repo_root / repo_relative_path
    if not absolute_path.is_file():
        raise FileNotFoundError(f"registered path is missing: {repo_relative_path}")
    return absolute_path


REPO_ROOT = require_repo_root()

STEP_ARTIFACTS = (
    StepArtifact(
        label="type2 non-model objects",
        step_path=Path("examples/type2/artifacts/type2_non_model_objects.step"),
        generator_source=Path("examples/type2/generate_non_model_step.py"),
    ),
)


In [2]:
tracked_result = subprocess.run(
    ["git", "ls-files", "*.step", "*.stp"],
    cwd=REPO_ROOT,
    check=True,
    capture_output=True,
    text=True,
)
TRACKED_STEP_PATHS = tuple(Path(line) for line in tracked_result.stdout.splitlines() if line)
REGISTERED_STEP_PATHS = tuple(artifact.step_path for artifact in STEP_ARTIFACTS)

unregistered_step_paths = tuple(sorted(set(TRACKED_STEP_PATHS) - set(REGISTERED_STEP_PATHS), key=str))
stale_registered_step_paths = tuple(sorted(set(REGISTERED_STEP_PATHS) - set(TRACKED_STEP_PATHS), key=str))
if unregistered_step_paths or stale_registered_step_paths:
    raise RuntimeError(
        "STEP notebook registry mismatch: "
        f"unregistered={unregistered_step_paths}, stale={stale_registered_step_paths}"
    )

for artifact in STEP_ARTIFACTS:
    require_file(REPO_ROOT, artifact.step_path)
    require_file(REPO_ROOT, artifact.generator_source)

print("registered STEP artifacts:")
for artifact in STEP_ARTIFACTS:
    print(f"- {artifact.label}: {artifact.step_path} (generator: {artifact.generator_source})")


registered STEP artifacts:
- type2 non-model objects: examples/type2/artifacts/type2_non_model_objects.step (generator: examples/type2/generate_non_model_step.py)


## type2 non-model objects

In [3]:
TYPE2_NON_MODEL_OBJECTS = STEP_ARTIFACTS[0]
type2_non_model_objects = bd.import_step(require_file(REPO_ROOT, TYPE2_NON_MODEL_OBJECTS.step_path))
show(
    type2_non_model_objects,
    names=[TYPE2_NON_MODEL_OBJECTS.label],
)


Using port 3939
Jupyter kernel running
Jupyter connection file path written to /home/harry/.ocpvscode
+++++++++c
